In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_excel('HDI_DATASET.xlsx',header=1)
df.head(33)

,Year,GDP per Capita (Constant 2015 US$),LIFE EXPECTANCY,MEAN YEARS OF SCHOOLING
0,1990,537.9,59,2.8
1,1991,531.9,59,2.9
2,1992,549.2,59,3.0
3,1993,563.4,60,3.1
4,1994,588.7,60,3.2
5,1995,620.7,61,3.3
6,1996,654.4,61,3.5
7,1997,667.6,61,3.6
8,1998,695.3,62,3.8
9,1999,742.5,62,4.0


In [ ]:
def calculate_crafts_income_utility(y, y_star=5120):

    # Define conditions for the piecewise segments
    conditions = [
        (y <= y_star),
        (y > y_star) & (y <= 2 * y_star),
        (y > 2 * y_star) & (y <= 3 * y_star),
        (y > 3 * y_star)
    ]

    # Define the corresponding formulas for each bracket
    choices = [
        y,
        y_star + 2 * (y - y_star) ** (1 / 2),
        y_star + 2 * (y_star) ** (1 / 2)
            + 3 * (y - 2 * y_star) ** (1 / 3),
        (
            y_star
            + 2 * (y_star) ** (1 / 2)
            + 3 * (y_star) ** (1 / 3)
            + 4 * (y - 3 * y_star) ** (1 / 4)
        )
    ]

    return np.select(conditions, choices, default=np.nan)

In [ ]:
# 1. Clean raw values (Run this BEFORE any normalization)
df.columns = df.columns.str.strip()


In [ ]:
# SELECTING THE COLUMNS TO BE NORMALISED
target_columns=['Human Development Index (HDI)','GNI per Capita (current,US$)','LIFE EXPECTANCY','EXPECTED YEARS OF SCHOOLING','MEAN YEARS OF SCHOOLING','INFANT MORTALITY(/1000 infants)']

In [ ]:
# 2. Compute the piecewise Adjusted Income on RAW GDP values
df["adjusted_income"] = calculate_crafts_income_utility(df['GDP per Capita (Constant 2015 US$)'], y_star=5120)


In [ ]:
# 3. Apply Fixed Goalpost Normalization to ALL variables
df["health_index"] = (df["LIFE EXPECTANCY"] - 25) / (85 - 25)
df["education_index"] = (df["MEAN YEARS OF SCHOOLING"] - 0) / (15 - 0)
df["income_index"] = (df["adjusted_income"] - 200) / (5385 - 200)
df.head(33)

,Year,GDP per Capita (Constant 2015 US$),LIFE EXPECTANCY,MEAN YEARS OF SCHOOLING,adjusted_income,health_index,education_index,income_index
0,1990,537.9,59,2.8,537.9,0.566667,0.186667,0.065169
1,1991,531.9,59,2.9,531.9,0.566667,0.193333,0.064012
2,1992,549.2,59,3.0,549.2,0.566667,0.200000,0.067348
3,1993,563.4,60,3.1,563.4,0.583333,0.206667,0.070087
4,1994,588.7,60,3.2,588.7,0.583333,0.213333,0.074966
5,1995,620.7,61,3.3,620.7,0.600000,0.220000,0.081138
6,1996,654.4,61,3.5,654.4,0.600000,0.233333,0.087637
7,1997,667.6,61,3.6,667.6,0.600000,0.240000,0.090183
8,1998,695.3,62,3.8,695.3,0.616667,0.253333,0.095526
9,1999,742.5,62,4.0,742.5,0.616667,0.266667,0.104629


In [ ]:
# 4. Calculate the true, unskewed Historical HDI
df["HDI_2"] = (
    df["health_index"] * df["education_index"] * df["income_index"]
) ** (1 / 3)
df.head(33)

,Year,GDP per Capita (Constant 2015 US$),LIFE EXPECTANCY,MEAN YEARS OF SCHOOLING,adjusted_income,health_index,education_index,income_index,HDI_2
0,1990,537.9,59,2.8,537.9,0.566667,0.186667,0.065169,0.190317
1,1991,531.9,59,2.9,531.9,0.566667,0.193333,0.064012,0.191410
2,1992,549.2,59,3.0,549.2,0.566667,0.200000,0.067348,0.196892
3,1993,563.4,60,3.1,563.4,0.583333,0.206667,0.070087,0.203677
4,1994,588.7,60,3.2,588.7,0.583333,0.213333,0.074966,0.210514
5,1995,620.7,61,3.3,620.7,0.600000,0.220000,0.081138,0.220428
6,1996,654.4,61,3.5,654.4,0.600000,0.233333,0.087637,0.230642
7,1997,667.6,61,3.6,667.6,0.600000,0.240000,0.090183,0.235051
8,1998,695.3,62,3.8,695.3,0.616667,0.253333,0.095526,0.246200
9,1999,742.5,62,4.0,742.5,0.616667,0.266667,0.104629,0.258161


In [ ]:
# NON-INCOME HDI 2
# Geometric mean of Health and Education
df["NIHDI2"] = (
    df["health_index"] * df["education_index"]
) ** (1 / 2)

# Display the result
df[["Year", "health_index", "education_index", "NIHDI2"]].head(33)

,Year,health_index,education_index,NIHDI2
0,1990,0.566667,0.186667,0.325235
1,1991,0.566667,0.193333,0.330992
2,1992,0.566667,0.200000,0.336650
3,1993,0.583333,0.206667,0.347211
4,1994,0.583333,0.213333,0.352767
5,1995,0.600000,0.220000,0.363318
6,1996,0.600000,0.233333,0.374166
7,1997,0.600000,0.240000,0.379473
8,1998,0.616667,0.253333,0.395250
9,1999,0.616667,0.266667,0.405518


In [ ]:
ts2 = df[[
    "Year",
    "HDI_2",
    "NIHDI2"
]].copy()

ts2.to_csv("ts_HDI2.csv", index=False)